In [1]:
# =====================================================
# STEP 1: Import Libraries
# =====================================================

import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    f1_score
)

from sklearn.feature_selection import (
    SelectKBest,
    chi2,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier

from scipy.sparse import hstack

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Amazon_Reviews.csv to Amazon_Reviews.csv


In [3]:
import pandas as pd

df = pd.read_csv(
    "Amazon_Reviews.csv",
    engine="python",
    encoding="latin1",
    on_bad_lines="skip"
)

In [4]:
print(df.head())

      Reviewer Name                     Profile Link Country Review Count  \
0        Eugene ath  /users/66e8185ff1598352d6b3701a      US     1 review   
1  Daniel ohalloran  /users/5d75e460200c1f6a6373648c      GB    9 reviews   
2          p fisher  /users/546cfcf1000064000197b88f      GB   90 reviews   
3         Greg Dunn  /users/62c35cdbacc0ea0012ccaffa      AU    5 reviews   
4     Sheila Hannah  /users/5ddbe429478d88251550610e      GB    8 reviews   

                Review Date                  Rating  \
0  2024-09-16T13:44:26.000Z  Rated 1 out of 5 stars   
1  2024-09-16T18:26:46.000Z  Rated 1 out of 5 stars   
2  2024-09-16T21:47:39.000Z  Rated 1 out of 5 stars   
3  2024-09-17T07:15:49.000Z  Rated 1 out of 5 stars   
4  2024-09-16T18:37:17.000Z  Rated 1 out of 5 stars   

                                      Review Title  \
0       A Store That Doesn't Want to Sell Anything   
1         Had multiple orders one turned up andâ¦   
2                      I informed these repr

In [5]:
print(df.columns.tolist())

['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']


In [6]:
print(df.shape)

(21214, 9)


In [9]:
df.dropna(inplace=True)

df.head()

,Review Text,Rating
0,"I registered on the website, tried to order a ...",Rated 1 out of 5 stars
1,Had multiple orders one turned up and driver h...,Rated 1 out of 5 stars
2,I informed these reprobates that I WOULD NOT B...,Rated 1 out of 5 stars
3,I have bought from Amazon before and no proble...,Rated 1 out of 5 stars
4,If I could give a lower rate I would! I cancel...,Rated 1 out of 5 stars


In [10]:


df["Rating"] = (
    df["Rating"]
    .astype(str)
    .str.extract(r'(\d+\.?\d*)')[0]
)

df["Rating"] = pd.to_numeric(
    df["Rating"],
    errors='coerce'
)

df.dropna(subset=["Rating"], inplace=True)

df["Rating"] = df["Rating"].astype(int)

print(df["Rating"].unique())

[1 5 2 4 3]


In [11]:
def clean_text(text):

    text=str(text)

    text=text.lower()

    text=re.sub(r"http\S+"," ",text)

    text=re.sub(r"[^a-zA-Z ]"," ",text)

    text=re.sub(r"\s+"," ",text)

    return text.strip()


df["Clean_Text"] = df["Review Text"].apply(clean_text)

In [12]:


negation_words = [
    "not","never","no",
    "nothing","none",
    "cannot","can't",
    "won't","don't",
    "isn't"
]


def sentence_length(text):

    return len(text.split())


def punctuation_density(text):

    punct_count=sum(
        1 for c in text
        if c in string.punctuation
    )

    total=max(len(text),1)

    return punct_count/total


def avg_word_length(text):

    words=text.split()

    if len(words)==0:
        return 0

    return np.mean(
        [len(word)
        for word in words]
    )


def negation_presence(text):

    words=text.split()

    return int(
        any(
            word in negation_words
            for word in words
        )
    )


df["sentence_length"]=df["Review Text"].apply(sentence_length)

df["punct_density"]=df["Review Text"].apply(punctuation_density)

df["avg_word_length"]=df["Review Text"].apply(avg_word_length)

df["negation"]=df["Review Text"].apply(negation_presence)

df.head()

,Review Text,Rating,Clean_Text,sentence_length,punct_density,avg_word_length,negation
0,"I registered on the website, tried to order a ...",1,i registered on the website tried to order a l...,106,0.040678,4.575472,1
1,Had multiple orders one turned up and driver h...,1,had multiple orders one turned up and driver h...,53,0.020478,4.547170,1
2,I informed these reprobates that I WOULD NOT B...,1,i informed these reprobates that i would not b...,122,0.016207,4.065574,1
3,I have bought from Amazon before and no proble...,1,i have bought from amazon before and no proble...,82,0.017778,4.487805,1
4,If I could give a lower rate I would! I cancel...,1,if i could give a lower rate i would i cancell...,100,0.018587,4.380000,1


In [13]:
tfidf=TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_text=tfidf.fit_transform(
    df["Clean_Text"]
)

print(X_text.shape)

(21055, 5000)


In [14]:
def create_sentiment(rating):

    if rating <= 2:
        return "Negative"

    elif rating == 3:
        return "Neutral"

    else:
        return "Positive"


df["Sentiment"] = df["Rating"].apply(create_sentiment)

print(df["Sentiment"].value_counts())

print(df.columns.tolist())

Sentiment
Negative    14350
Positive     5820
Neutral       885
Name: count, dtype: int64
['Review Text', 'Rating', 'Clean_Text', 'sentence_length', 'punct_density', 'avg_word_length', 'negation', 'Sentiment']


In [15]:
extra_features=df[
    [
        "sentence_length",
        "punct_density",
        "avg_word_length",
        "negation"
    ]
].values


X=hstack(
    [X_text,extra_features]
)

y=df["Sentiment"]

In [16]:
encoder=LabelEncoder()

y=encoder.fit_transform(y)

print(
encoder.classes_
)

['Negative' 'Neutral' 'Positive']


In [17]:
X_train,X_test,y_train,y_test=(
train_test_split(
    X,
    y,
    test_size=.2,
    random_state=42,
    stratify=y
)
)